In [22]:
# Data Manipulation and Math
import random
import math
import pandas as pd
import networkx as nx

# System Performance and Benchmarking
import psutil
import time
import os
import json   # Only if exporting summary to JSON

#Visualisation
import matplotlib.pyplot as plt

Implementation of Simulated Annealing. 
The implementation includes:

- init – Initializes the graph and stores the list of vertices.
- cut_value – Computes the cut value of a given partition.
- neighbour – Generates a neighboring solution by flipping the partition assignment of a randomly selected vertex.
- optimal_cut – Computes the optimal cut value using exhaustive search for comparison.
- optimize – Executes the simulated annealing algorithm, applies the cooling schedule and acceptance criterion, tracks the best solution, computes performance metrics, and returns the final results.

In [28]:
class SimulatedAnnealing:

    def __init__(self, graph):

        self.graph = graph
        self.nodes = list(graph.nodes())

    def cut_value(self, partition):

        cut = 0

        for u, v in self.graph.edges():

            if partition[u] != partition[v]:
                cut += 1

        return cut

    def neighbour(self, partition):

        new_partition = partition.copy()

        node = random.choice(self.nodes)

        new_partition[node] = 1 - new_partition[node]

        return new_partition

    def optimal_cut(self):

        n = len(self.nodes)

        best = 0

        for mask in range(1 << n):

            partition = {}

            for i, node in enumerate(self.nodes):
                partition[node] = (mask >> i) & 1

            best = max(best, self.cut_value(partition))

        return best

    def optimize(
        self,
        initial_partition,
        initial_temp=100,
        cooling_rate=0.995,
        min_temp=0.001,
        max_iterations=1000
    ):

        current = initial_partition.copy()
        current_cut = self.cut_value(current)

        best = current.copy()
        best_cut = current_cut

        cut_history = [current_cut]

        temperature = initial_temp

        iteration = 0

        while temperature > min_temp and iteration < max_iterations:

            candidate = self.neighbour(current)
            candidate_cut = self.cut_value(candidate)

            delta = candidate_cut - current_cut

            if delta > 0:

                current = candidate
                current_cut = candidate_cut

            else:

                probability = math.exp(delta / temperature)

                if random.random() < probability:

                    current = candidate
                    current_cut = candidate_cut

            cut_history.append(current_cut)

            if current_cut > best_cut:

                best = current.copy()
                best_cut = current_cut

            temperature *= cooling_rate
            iteration += 1

        average_cut = sum(cut_history) / len(cut_history)

        optimal_cut = self.optimal_cut()

        approximation_ratio = (
            best_cut / optimal_cut
            if optimal_cut > 0
            else 0
        )

        return {

            "best_partition": best,
            "best_cut": best_cut,
            "average_cut": average_cut,
            "approximation_ratio": approximation_ratio

        }

Generates a random d-regular graph, where every node has the same specified degree (default = 3), and returns the generated graph.

In [29]:
def generate_regular_graph(num_nodes, degree=3):

    graph = nx.random_regular_graph(degree, num_nodes)

    return graph

Creates and returns a random initial partition by assigning each vertex to one of two partitions.

In [30]:
def initialize_partition(graph):

    partition = {}

    for node in graph.nodes():
        partition[node] = random.randint(0, 1)

    return partition

Benchmarks the simulated annealing algorithm across different graph sizes and multiple trials, recording :
- execution time
- CPU time
- throughput
- memory usage
- best cut
- average cut
- approximation ratio

and returns the collected performance results.

In [31]:
def benchmark_simulated_annealing(
    graph_sizes,
    trials=10
):

    process = psutil.Process(os.getpid())

    results = []

    for size in graph_sizes:

        print(f"Benchmarking {size} nodes...")

        for trial in range(1, trials + 1):

            graph = generate_regular_graph(size)

            partition = initialize_partition(graph)

            sa = SimulatedAnnealing(graph)

            cpu_start = time.process_time()
            wall_start = time.perf_counter()

            output = sa.optimize(partition)

            wall_end = time.perf_counter()
            cpu_end = time.process_time()

            runtime = wall_end - wall_start
            cpu_time = cpu_end - cpu_start

            throughput = size / runtime

            peak_ram = process.memory_info().rss / (1024 * 1024)

            results.append({

                "Nodes": size,
                "Trial": trial,
                "Runtime (s)": runtime,
                "CPU Time (s)": cpu_time,
                "Throughput": throughput,
                "Peak RAM (MB)": peak_ram,
                "Best Cut": output["best_cut"],
                "Average Cut": output["average_cut"],
                "Approximation Ratio": output["approximation_ratio"]

            })

    return results

Defines the list of graph sizes to be used for benchmarking.

In [32]:
graph_sizes = [4, 6, 8, 10, 12]

results = benchmark_simulated_annealing(
    graph_sizes=graph_sizes,
    trials=10
)

Benchmarking 4 nodes...
Benchmarking 6 nodes...
Benchmarking 8 nodes...
Benchmarking 10 nodes...
Benchmarking 12 nodes...


Converts the benchmark results into a DataFrame and displays the collected performance metrics in tabular form.

In [33]:
df = pd.DataFrame(results)

display(df)

,Nodes,Trial,Runtime (s),CPU Time (s),Throughput,Peak RAM (MB),Best Cut,Average Cut,Approximation Ratio
0,4,1,0.010982,0.000000,364.239013,153.742188,4,3.243756,1.0
1,4,2,0.010206,0.015625,391.926318,153.742188,4,3.176823,1.0
2,4,3,0.010130,0.015625,394.862834,153.742188,4,3.236763,1.0
3,4,4,0.009925,0.015625,403.038913,153.742188,4,3.199800,1.0
4,4,5,0.010561,0.000000,378.762771,153.742188,4,3.250749,1.0
5,4,6,0.011424,0.015625,350.149251,153.742188,4,3.235764,1.0
6,4,7,0.012025,0.015625,332.637566,153.742188,4,3.228771,1.0
7,4,8,0.010897,0.000000,367.063401,153.742188,4,3.231768,1.0
8,4,9,0.010251,0.015625,390.194414,153.742188,4,3.295704,1.0
9,4,10,0.010499,0.015625,380.974151,153.742188,4,3.199800,1.0


Groups the benchmark results by graph size and computes the mean and standard deviation for all performance metrics.

In [34]:
summary_df = (
    df.groupby("Nodes")
      .agg({
          "Runtime (s)": ["mean", "std"],
          "CPU Time (s)": ["mean", "std"],
          "Throughput": ["mean", "std"],
          "Peak RAM (MB)": ["mean", "std"],
          "Best Cut": ["mean", "std"],
          "Average Cut": ["mean", "std"],
          "Approximation Ratio": ["mean", "std"]
      })
)

summary_df

Runtime (s)           CPU Time (s)            Throughput             \
             mean       std         mean       std        mean        std   
Nodes                                                                       
4        0.010690  0.000652     0.010937  0.007548  375.384863  21.952917   
6        0.014247  0.000907     0.010937  0.007548  422.710857  27.466839   
8        0.020074  0.000672     0.020313  0.007548  398.942062  13.798617   
10       0.038954  0.001270     0.040625  0.008069  256.962041   8.495264   
12       0.127334  0.004215     0.121875  0.012325   94.335045   3.186430   

      Peak RAM (MB)      Best Cut           Average Cut            \
               mean  std     mean       std        mean       std   
Nodes                                                               
4        153.742188  0.0      4.0  0.000000    3.229970  0.032783   
6        153.742188  0.0      7.0  0.000000    5.047752  0.086054   
8        153.742188  0.0      9.8  0.632456    6.642657  0.134331   
10       153.742188  0.0     12.9  0.316228    8.411189  0.132929   
12       153.742188  0.0     15.5  0.527046   10.153147  0.197603   

      Approximation Ratio       
                     mean  std  
Nodes                           
4                     1.0  0.0  
6                     1.0  0.0  
8                     1.0  0.0  
10                    1.0  0.0  
12                    1.0  0.0

Saves the detailed benchmark results and summary statistics as CSV files for further analysis and reporting.

In [37]:
df.to_csv(
    "simulated_annealing_results.csv",
    index=False
)

summary_df.to_csv(
    "simulated_annealing_summary.csv"
)

Converts and saves the summary results in JSON format.

In [43]:
json_df = summary_df.copy()

json_df.columns = [
    f"{col[0]}_{col[1]}"
    for col in json_df.columns
]

json_df = json_df.reset_index()

json_df.to_json(
    "simulated_annealing_summary.json",
    orient="records",
    indent=4
)